# Energy Asset Monitoring Framework

## Synthetic Dataset Generation

This notebook generates a synthetic dataset representing the operation of a Compressed Air Energy Storage (CAES) pilot plant.

The objective is to create a realistic time-series dataset that can be used to demonstrate a complete energy analytics workflow, including:

- data quality assessment
- operational state detection
- KPI calculation
- interactive reporting

The dataset is intentionally synthetic and does not represent any confidential or proprietary industrial data.

## Engineering Assumptions

This synthetic dataset is not intended to reproduce the exact thermodynamic behaviour of a CAES plant.

Instead, the following assumptions are adopted:

- Operating modes are predefined.
- Pressure evolves according to a first-order dynamic.
- Temperature tends towards an operating-mode-dependent equilibrium.
- Compressor and turbine power are simplified empirical relationships.
- Random noise emulates measurement uncertainty.

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"

from pathlib import Path

# =============================================================================
# Project directories
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"

SAVE_HTML = True
SHOW_FIGURE = False

In [2]:
# =============================================================================
# Simulation parameters
# =============================================================================

RANDOM_SEED = 42
SIMULATION_HOURS = 48
SAMPLING_TIME = "10s"
START_DATE = "2026-01-01"

np.random.seed(RANDOM_SEED)

In [3]:
# =============================================================================
# Timeline
# =============================================================================

time_index = pd.date_range(
    start=START_DATE,
    periods=int(SIMULATION_HOURS * 3600 / 10),
    freq=SAMPLING_TIME
)

df = pd.DataFrame(index=time_index)
df.index.name = "timestamp"

print(f"Samples: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

df.head()

Samples: 17,280
Start: 2026-01-01 00:00:00
End:   2026-01-02 23:59:50


""
timestamp
2026-01-01 00:00:00
2026-01-01 00:00:10
2026-01-01 00:00:20
2026-01-01 00:00:30
2026-01-01 00:00:40


## Operating Profile

The first step is to define the operating state of the plant.

Instead of generating each signal independently, the synthetic dataset is built around a predefined operating profile. This ensures that all variables remain physically consistent throughout the simulation.

The operating modes considered are:

- STANDBY
- CHARGING
- DISCHARGING

In [4]:
# =============================================================================
# Operating profile
# =============================================================================

df["operating_mode"] = "STANDBY"

schedule = [
    (6, 12, "CHARGING"),
    (15, 19, "DISCHARGING"),
    (22, 26, "CHARGING"),
    (30, 34, "DISCHARGING"),
    (39, 41, "CHARGING")
]

simulation_start = df.index[0]

for start_hour, end_hour, state in schedule:

    start_time = simulation_start + pd.Timedelta(hours=start_hour)
    end_time = simulation_start + pd.Timedelta(hours=end_hour)

    mask = (df.index >= start_time) & (df.index < end_time)

    df.loc[mask, "operating_mode"] = state

df["operating_mode"].value_counts()

operating_mode
STANDBY        10080
CHARGING        4320
DISCHARGING     2880
Name: count, dtype: int64

In [10]:
# =============================================================================
# Cycle identification
# =============================================================================

cycle_id = []

current_cycle = 1
previous_mode = "STANDBY"

for mode in df["operating_mode"]:

    # New cycle starts when charging begins
    if (
        mode == "CHARGING"
        and previous_mode != "CHARGING"
    ):
        current_cycle += 1

    cycle_id.append(current_cycle)

    previous_mode = mode

df["cycle_id"] = cycle_id

In [11]:
state_map = {
    "STANDBY": 0,
    "CHARGING": 1,
    "DISCHARGING": 2
}

fig = px.line(x=df.index,
              y=df["operating_mode"].map(state_map),
              labels={"x": "Time", "y": "Operating mode"},
              title="Operating profile")

fig.update_yaxes(tickvals=[0,1,2],
                 ticktext=["Standby","Charging","Discharging"])

if SAVE_HTML:
    fig.write_html(FIGURES_DIR / "operating_profile.html",
                   include_plotlyjs="cdn")

if SHOW_FIGURE:
    fig.show()

In [12]:
simulation_info = {
    "simulation_hours": 48,
    "sampling_time": "10 s",
    "number_of_samples": len(df),
    "charging_cycles": (df["operating_mode"] == "CHARGING").sum(),
    "discharging_cycles": (df["operating_mode"] == "DISCHARGING").sum()
}

## Simplified Physical Model

The plant behaviour is represented through simplified physical relationships.

The objective is not to reproduce a detailed thermodynamic model, but to generate realistic time-series data with consistent relationships between:

- operating mode
- storage pressure
- temperature evolution
- energy flows

The model considers three main operating conditions:

- Charging: air compression and energy storage
- Discharging: energy recovery from stored air
- Stand-by: natural pressure losses and thermal relaxation

In [13]:
# =============================================================================
# Simplified plant parameters
# =============================================================================

P_INITIAL = 250      # bar
P_MIN = 100          # bar
P_MAX = 400          # bar
CHARGING_RATE = 0.08
DISCHARGING_RATE = 0.10
LEAKAGE_RATE = 0.002

T_INITIAL = 25       # °C
THERMAL_RELAXATION = 0.01

In [14]:
# =============================================================================
# Ambient conditions
# =============================================================================
df["ambient_temperature"] = (20 + 3 * np.sin(np.linspace(0,2*np.pi,len(df))))
df.head()

,operating_mode,cycle_id,ambient_temperature
timestamp,,,
2026-01-01 00:00:00,STANDBY,1,20.000000
2026-01-01 00:00:10,STANDBY,1,20.001091
2026-01-01 00:00:20,STANDBY,1,20.002182
2026-01-01 00:00:30,STANDBY,1,20.003273
2026-01-01 00:00:40,STANDBY,1,20.004364


In [15]:
def update_pressure(current_pressure,operating_mode):
    """
    Update the compressed air storage pressure using a simplified
    first-order dynamic model.

    The pressure evolution depends on the current operating condition:

    - CHARGING:
        Pressure increases as air is compressed into the storage system.
        The charging rate decreases as the pressure approaches the maximum
        storage limit.

    - DISCHARGING:
        Pressure decreases proportionally to the available stored pressure.

    - STANDBY:
        A small leakage term is applied to represent slow pressure losses.

    Parameters
    ----------
    current_pressure : float
        Current storage pressure [bar].

    operating_mode : str
        Current plant operating mode:
        "CHARGING", "DISCHARGING" or "STANDBY".

    Returns
    -------
    float
        Updated storage pressure [bar].

    Notes
    -----
    This model is intentionally simplified and does not reproduce the
    detailed thermodynamic behaviour of a real CAES plant.
    It is designed to generate physically consistent synthetic data.
    """

    if operating_mode == "CHARGING":
        delta_p = (CHARGING_RATE * (1 - current_pressure / P_MAX))

    elif operating_mode == "DISCHARGING":
        delta_p = (-DISCHARGING_RATE * (current_pressure / P_MAX))

    else:
        delta_p = -LEAKAGE_RATE

    new_pressure = current_pressure + delta_p

    return np.clip(new_pressure,P_MIN,P_MAX)
##
##--------------------------------------
def update_temperature(current_temperature,operating_mode,
                       ambient_temperature):
    """
    Update the storage temperature using a first-order thermal relaxation model.

    The storage temperature gradually approaches an equilibrium temperature
    defined by the operating mode:

    - CHARGING:
        Temperature increases due to compression heating.

    - DISCHARGING:
        Temperature decreases due to expansion cooling.

    - STANDBY:
        Temperature tends towards ambient conditions.

    Parameters
    ----------
    current_temperature : float
        Current storage temperature [°C].

    operating_mode : str
        Current plant operating mode:
        "CHARGING", "DISCHARGING" or "STANDBY".

    ambient_temperature : float
        External ambient temperature [°C].
                
    Returns
    -------
    float
        Updated storage temperature [°C].

    Notes
    -----
    The implemented equation represents a generic first-order dynamic:
        T_new = T_current + alpha * (T_equilibrium - T_current)
    where alpha represents the thermal response speed.

    The model is not intended to reproduce detailed heat transfer
    phenomena but only the expected qualitative behaviour.
    """

    if operating_mode == "CHARGING":
        target_temperature = 60

    elif operating_mode == "DISCHARGING":
        target_temperature = 10

    else:
        target_temperature = ambient_temperature

    return current_temperature + THERMAL_RELAXATION * (
        target_temperature - current_temperature
    )
##
##--------------------------------------
def update_compressor_power(storage_pressure, operating_mode):
    """
    Update the compressor electrical power.

    The compressor consumes electrical power only during the CHARGING mode.
    The required power increases with storage pressure, representing the
    higher compression effort needed as the storage approaches its maximum
    pressure.

    Parameters
    ----------
    storage_pressure : float
        Current storage pressure [bar].

    operating_mode : str
        Current plant operating mode:
        "CHARGING", "DISCHARGING" or "STANDBY".

    Returns
    -------
    float
        Compressor electrical power [kW].

    Notes
    -----
    The implemented model is intentionally simplified and is designed only
    to reproduce the expected qualitative behaviour of a CAES compressor.
    """
    COMPRESSOR_POWER_BASE = 100      # kW, lower value (low pressure)
    COMPRESSOR_POWER_MAX_DELTA = 40  # kW, max increment towards P_MAX
    
    if operating_mode != "CHARGING":
        return 0.0

    pressure_ratio = (storage_pressure - P_MIN) / (P_MAX - P_MIN)

    return (
        COMPRESSOR_POWER_BASE
        + COMPRESSOR_POWER_MAX_DELTA * pressure_ratio
    )
##
##--------------------------------------
def update_turbine_power(storage_pressure, operating_mode):
    """
    Update the turbine electrical power output.

    The turbine generates electrical power only during the DISCHARGING mode.
    The generated power increases with the available storage pressure,
    reflecting the larger amount of extractable energy.

    Parameters
    ----------
    storage_pressure : float
        Current storage pressure [bar].

    operating_mode : str
        Current plant operating mode:
        "CHARGING", "DISCHARGING" or "STANDBY".

    Returns
    -------
    float
        Turbine electrical power output [kW].

    Notes
    -----
    This simplified model is intended to generate realistic synthetic data
    rather than accurately reproduce turbine thermodynamics.
    """
    TURBINE_POWER_BASE = 50      # kW, lower value (low pressure)
    TURBINE_POWER_MAX_DELTA = 15 # kW, max increment towards P_MAX
    
    if operating_mode != "DISCHARGING":
        return 0.0

    pressure_ratio = (storage_pressure - P_MIN) / (P_MAX - P_MIN)

    return (
        TURBINE_POWER_BASE
        + TURBINE_POWER_MAX_DELTA * pressure_ratio
    )
##
##--------------------------------------
def update_mass_flow(operating_mode):
    """
    Update the air mass flow rate.

    The mass flow depends only on the current operating mode:

    - CHARGING:
        Positive mass flow representing air entering the storage.

    - DISCHARGING:
        Negative mass flow representing air leaving the storage.

    - STANDBY:
        Zero mass flow.

    A Gaussian random noise term is added to emulate measurement
    uncertainty and small process fluctuations.

    Parameters
    ----------
    operating_mode : str
        Current plant operating mode:
        "CHARGING", "DISCHARGING" or "STANDBY".

    Returns
    -------
    float
        Air mass flow rate [kg/s].

    Notes
    -----
    The model generates synthetic mass flow values representative of the
    operating condition and is not intended to simulate detailed fluid
    dynamics.
    """

    if operating_mode == "CHARGING":
        return 0.80#np.random.normal(0.80, 0.02)

    elif operating_mode == "DISCHARGING":
        return -0.55#np.random.normal(-0.55, 0.02)

    return 0.0
##
##--------------------------------------

In [16]:
pressure = []
temperature = []
#
current_pressure = P_INITIAL
current_temperature = T_INITIAL
#
for timestamp, row in df.iterrows():
    #
    mode = row["operating_mode"]
    #
    current_pressure = update_pressure(
        current_pressure,
        mode
    )
    #
    current_temperature = update_temperature(
        current_temperature,
        row["operating_mode"],
        row["ambient_temperature"]
    )
    #
    pressure.append(current_pressure)
    temperature.append(current_temperature)
#
df["tank_pressure"] = pressure
df["tank_temperature"] = temperature

In [17]:
compressor_power = []
turbine_power = []
mass_flow = []

for mode, pressure in zip(df["operating_mode"], df["tank_pressure"]):

    compressor_power.append(
        update_compressor_power(pressure, mode)
    )

    turbine_power.append(
        update_turbine_power(pressure, mode)
    )

    mass_flow.append(
        update_mass_flow(mode)
    )

df["compressor_power"] = compressor_power
df["turbine_power"] = turbine_power
df["mass_flow"] = mass_flow

In [18]:
def add_measurement_noise(df,
                          pressure_std=0.5,
                          temperature_std=0.3,
                          compressor_std=4,
                          turbine_std=2,
                          mass_flow_std=0.02):
    """
    Create a noisy copy of a synthetic CAES dataset.

    Random Gaussian measurement noise is added only when the corresponding
    equipment is operating. Zero-power conditions are preserved to avoid
    generating unrealistic negative or positive power values.

    Parameters
    ----------
    df : pandas.DataFrame
        Clean synthetic dataset.

    pressure_std : float
        Standard deviation of pressure measurement noise [bar].

    temperature_std : float
        Standard deviation of temperature measurement noise [°C].

    compressor_std : float
        Standard deviation of compressor power noise [kW].

    turbine_std : float
        Standard deviation of turbine power noise [kW].

    mass_flow_std : float
        Standard deviation of mass flow noise [kg/s].

    Returns
    -------
    pandas.DataFrame
        Copy of the input dataset with realistic measurement noise.
    """

    df_noisy = df.copy()

    # Continuous measurements
    df_noisy["tank_pressure"] += np.random.normal(
        0,
        pressure_std,
        len(df_noisy)
    )

    df_noisy["tank_temperature"] += np.random.normal(
        0,
        temperature_std,
        len(df_noisy)
    )

    # Compressor noise only during operation
    compressor_mask = df_noisy["compressor_power"] > 0

    df_noisy.loc[compressor_mask, "compressor_power"] += np.random.normal(
        0,
        compressor_std,
        compressor_mask.sum()
    )

    # Turbine noise only during operation
    turbine_mask = df_noisy["turbine_power"] > 0

    df_noisy.loc[turbine_mask, "turbine_power"] += np.random.normal(
        0,
        turbine_std,
        turbine_mask.sum()
    )

    # Mass flow noise only when air is moving
    flow_mask = df_noisy["mass_flow"] != 0

    df_noisy.loc[flow_mask, "mass_flow"] += np.random.normal(
        0,
        mass_flow_std,
        flow_mask.sum()
    )

    return df_noisy
##
##--------------------------------------

In [19]:
# =============================================================================
# Measurement noise
# =============================================================================

PRESSURE_NOISE_STD = 0.5      # bar
TEMPERATURE_NOISE_STD = 0.3   # °C
COMPRESSOR_NOISE_STD = 4
TURBINE_NOISE_STD = 2
#
df_clean = df.copy()
df_noisy = add_measurement_noise(df_clean, 
                                 pressure_std=PRESSURE_NOISE_STD,
                                 temperature_std=TEMPERATURE_NOISE_STD,
                                 compressor_std=COMPRESSOR_NOISE_STD,
                                 turbine_std=TURBINE_NOISE_STD)

## Anomaly Injection

The noisy dataset generated so far only contains Gaussian measurement noise, superimposed on an otherwise perfectly consistent signal. In a real monitoring system, sensors are also affected by more disruptive issues: temporary dropouts, isolated spurious readings, sensors that freeze on a fixed value, or slow calibration drifts.

To make the downstream Data Quality Assessment (notebook 2) meaningful, a small set of controlled and reproducible anomalies is injected into `df_noisy` only. `df_clean` remains untouched and continues to represent the ground-truth physical behaviour of the plant.

For simplicity, only two anomaly types are considered at the moment:

- **Missing values**: temporary sensor dropout windows.
- **Outliers**: isolated spurious readings, out of the expected measurement range.

In the future, other anomalies could be considered:

- **Stuck sensor**: a sensor freezing on a constant value for an extended period.
- **Sensor drift**: a slowly growing offset, representing calibration loss over time.

Every injected anomaly is recorded in an `anomaly_log`, saved alongside the dataset. This provides a ground truth of known issues that can be used in notebook 2 to validate how many anomalies the quality checks are actually able to detect.


In [20]:
# =============================================================================
# Anomaly injection functions
# =============================================================================

def inject_missing_values(df, columns, n_windows, window_length_range, rng):
    """
    Inject missing-value windows into selected columns.

    A number of time windows of random length are set to NaN in each
    of the specified columns, to emulate temporary sensor dropout.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to be modified (a copy is returned, the input is not
        mutated in place).

    columns : list of str
        Columns affected by missing-value windows.

    n_windows : int
        Number of dropout windows injected per column.

    window_length_range : tuple of int
        Minimum and maximum window length, in number of samples.

    rng : numpy.random.Generator
        Random number generator, for reproducibility.

    Returns
    -------
    df_out : pandas.DataFrame
        Dataset with injected missing values.

    log : list of dict
        Record of injected anomalies (column, position, type).
    """

    df_out = df.copy()
    log = []

    for column in columns:
        col_idx = df_out.columns.get_loc(column)

        for _ in range(n_windows):
            length = int(rng.integers(*window_length_range))
            start = int(rng.integers(0, len(df_out) - length))
            end = start + length

            df_out.iloc[start:end, col_idx] = np.nan

            log.append({
                "type": "missing_values",
                "column": column,
                "start_index": start,
                "end_index": end,
                "n_samples": length
            })

    return df_out, log
##
##--------------------------------------
def inject_outliers(df, columns, n_outliers, magnitude_range, rng):
    """
    Inject isolated outlier spikes into selected columns.

    Each outlier is generated by adding a random offset, expressed as
    a multiple of the column's standard deviation, to a single sample.
    Outliers are injected independently of the operating mode, so that
    they may also produce operating-mode inconsistencies (e.g. a
    non-zero compressor power reading while the plant is on standby).

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to be modified (a copy is returned).

    columns : list of str
        Columns affected by outliers.

    n_outliers : int
        Number of outliers injected per column.

    magnitude_range : tuple of float
        Minimum and maximum outlier magnitude, expressed as a multiple
        of the column standard deviation.

    rng : numpy.random.Generator
        Random number generator, for reproducibility.

    Returns
    -------
    df_out : pandas.DataFrame
        Dataset with injected outliers.

    log : list of dict
        Record of injected anomalies (column, position, magnitude).
    """

    df_out = df.copy()
    log = []

    for column in columns:
        col_idx = df_out.columns.get_loc(column)
        col_std = df_out[column].std()

        indices = rng.choice(len(df_out), size=n_outliers, replace=False)

        for idx in indices:
            magnitude = rng.uniform(*magnitude_range)
            sign = rng.choice([-1, 1])

            df_out.iloc[int(idx), col_idx] += sign * magnitude * col_std

            log.append({
                "type": "outlier",
                "column": column,
                "index": int(idx),
                "magnitude_std": float(magnitude),
                "sign": int(sign)
            })

    return df_out, log
##
##--------------------------------------
# FOR THE FUTURE
def inject_stuck_sensor(df, column, n_windows, window_length_range, rng):
    """
    Inject stuck-sensor windows into a selected column.

    Within each window, the sensor value is frozen at the value
    observed at the start of the window, emulating a sensor that stops
    updating while still reporting a plausible value.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to be modified (a copy is returned).

    column : str
        Column affected by the stuck-sensor condition.

    n_windows : int
        Number of stuck-sensor windows injected.

    window_length_range : tuple of int
        Minimum and maximum window length, in number of samples.

    rng : numpy.random.Generator
        Random number generator, for reproducibility.

    Returns
    -------
    df_out : pandas.DataFrame
        Dataset with injected stuck-sensor windows.

    log : list of dict
        Record of injected anomalies (column, position, frozen value).
    """

    df_out = df.copy()
    log = []
    col_idx = df_out.columns.get_loc(column)

    for _ in range(n_windows):
        length = int(rng.integers(*window_length_range))
        start = int(rng.integers(0, len(df_out) - length))
        end = start + length

        frozen_value = df_out.iloc[start, col_idx]
        df_out.iloc[start:end, col_idx] = frozen_value

        log.append({
            "type": "stuck_sensor",
            "column": column,
            "start_index": start,
            "end_index": end,
            "n_samples": length,
            "frozen_value": float(frozen_value)
        })

    return df_out, log
##
##--------------------------------------
# FOR THE FUTURE
def inject_sensor_drift(df, column, start_fraction, total_offset):
    """
    Inject a slow linear sensor drift starting at a given point in time.

    From ``start_fraction`` of the dataset length onward, a linearly
    increasing offset is added to the column, reaching ``total_offset``
    at the end of the dataset. This emulates a gradual loss of sensor
    calibration.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to be modified (a copy is returned).

    column : str
        Column affected by the drift.

    start_fraction : float
        Fraction (0-1) of the dataset length at which the drift begins.

    total_offset : float
        Offset reached at the end of the dataset, in the column's
        physical unit.

    Returns
    -------
    df_out : pandas.DataFrame
        Dataset with injected sensor drift.

    log : list of dict
        Record of the injected anomaly (column, position, total offset).
    """

    df_out = df.copy()
    n = len(df_out)
    start_idx = int(n * start_fraction)

    drift = np.zeros(n)
    drift[start_idx:] = np.linspace(0, total_offset, n - start_idx)

    df_out[column] += drift

    log = [{
        "type": "sensor_drift",
        "column": column,
        "start_index": start_idx,
        "end_index": n - 1,
        "total_offset": total_offset
    }]

    return df_out, log


In [21]:
# =============================================================================
# Anomaly injection parameters
# =============================================================================

ANOMALY_SEED = 123
rng = np.random.default_rng(ANOMALY_SEED)

MISSING_COLUMNS = ["tank_temperature", "ambient_temperature"]
N_MISSING_WINDOWS = 2
MISSING_WINDOW_LENGTH_RANGE = (10, 60)    # samples (~100-600 s)

OUTLIER_COLUMNS = ["tank_pressure", "compressor_power"]
N_OUTLIERS_PER_COLUMN = 10
OUTLIER_MAGNITUDE_RANGE = (5, 10)         # multiples of column std

In [22]:
# =============================================================================
# Apply anomaly injection to the noisy dataset
# =============================================================================
# df_clean is intentionally left untouched: it represents the
# ground-truth physical behaviour of the plant.

anomaly_log = []

df_noisy, log_missing = inject_missing_values(
    df_noisy,
    columns=MISSING_COLUMNS,
    n_windows=N_MISSING_WINDOWS,
    window_length_range=MISSING_WINDOW_LENGTH_RANGE,
    rng=rng
)
anomaly_log += log_missing

df_noisy, log_outliers = inject_outliers(
    df_noisy,
    columns=OUTLIER_COLUMNS,
    n_outliers=N_OUTLIERS_PER_COLUMN,
    magnitude_range=OUTLIER_MAGNITUDE_RANGE,
    rng=rng
)
anomaly_log += log_outliers

anomaly_log_df = pd.DataFrame(anomaly_log)

print(f"Total injected anomalies: {len(anomaly_log_df)}")
anomaly_log_df


Total injected anomalies: 24


,type,column,start_index,end_index,n_samples,index,magnitude_std,sign
0,missing_values,tank_temperature,11784.0,11794.0,10.0,NaN,NaN,NaN
1,missing_values,tank_temperature,927.0,966.0,39.0,NaN,NaN,NaN
2,missing_values,ambient_temperature,3795.0,3850.0,55.0,NaN,NaN,NaN
3,missing_values,ambient_temperature,3181.0,3203.0,22.0,NaN,NaN,NaN
4,outlier,tank_pressure,NaN,NaN,NaN,7765.0,8.707335,-1.0
5,outlier,tank_pressure,NaN,NaN,NaN,7797.0,8.149701,-1.0
6,outlier,tank_pressure,NaN,NaN,NaN,6007.0,6.159541,1.0
7,outlier,tank_pressure,NaN,NaN,NaN,4778.0,8.995626,-1.0
8,outlier,tank_pressure,NaN,NaN,NaN,15951.0,6.157778,1.0
9,outlier,tank_pressure,NaN,NaN,NaN,3038.0,5.829520,-1.0


In [23]:
# =============================================================================
# Save anomaly log
# =============================================================================
# Ground truth of injected anomalies, to be used in notebook 2 to
# evaluate how many of them are actually detected by the quality checks.

anomaly_log_file = REPORTS_DIR / "injected_anomalies_log.csv"

anomaly_log_df.to_csv(
    anomaly_log_file,
    index=False
)

print(f"Anomaly log saved to:\n{anomaly_log_file}")


Anomaly log saved to:
c:\Users\ma_gatti\Documents\Python\EnergyAnalyticsFramework\reports\injected_anomalies_log.csv


In [24]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
##
##--------------------------------------
def plot_caes_timeseries(df, title="CAES Synchronized Signals", 
                         save_html = False, fig_name = 'Signals_', show_figure = True):
    """
    Plot the main CAES operating signals in four synchronized subplots.

    The figure displays:

    1. Storage pressure
    2. Storage temperature
    3. Air mass flow rate
    4. Compressor and turbine electrical power

    All subplots share the same time axis and a synchronized hover cursor,
    allowing easy inspection of the operating conditions at any instant.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the CAES time series.

        Required columns are:
        - tank_pressure
        - tank_temperature
        - mass_flow
        - compressor_power
        - turbine_power

    title : str, optional
        Figure title.

    save_html : bool, optional
        Default is false.

    fig_name : str, optional
        Filename of the figure.

    show_figure : bool, optional
        Default is true.
        
    Returns
    -------
    plotly.graph_objects.Figure
        Interactive Plotly figure.

    Notes
    -----
    The function is independent of the data generation process and can be
    used with clean, noisy or fault-injected datasets, provided the required
    columns are available.
    """

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(
            "Tank Pressure [bar]",
            "Tank Temperature [°C]",
            "Mass Flow [kg/s]",
            "Power [kW]"
        ),
        row_heights=[0.25, 0.25, 0.25, 0.25]
    )

    # ------------------------------------------------------------------
    # Pressure
    # ------------------------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["tank_pressure"],
            name="Pressure",
            line=dict(color="royalblue")
        ),
        row=1,
        col=1
    )

    # ------------------------------------------------------------------
    # Temperature
    # ------------------------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["tank_temperature"],
            name="Temperature",
            line=dict(color="firebrick")
        ),
        row=2,
        col=1
    )

    # ------------------------------------------------------------------
    # Mass flow
    # ------------------------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["mass_flow"],
            name="Mass flow",
            line=dict(color="seagreen")
        ),
        row=3,
        col=1
    )

    # ------------------------------------------------------------------
    # Power
    # ------------------------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["compressor_power"],
            name="Compressor power",
            line=dict(color="darkorange")
        ),
        row=4,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["turbine_power"],
            name="Turbine power",
            line=dict(color="purple")
        ),
        row=4,
        col=1
    )

    # ------------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------------
    fig.update_layout(
        height=900,
        title=title,
        hovermode="x",
        showlegend=True
    )

    # Synchronize hover line across all panels
    fig.update_traces(xaxis="x4")

    fig.update_xaxes(
        showspikes=True,
        spikemode="across",
        spikesnap="cursor",
        spikethickness=1,
        spikedash="dot"
    )

    fig.update_xaxes(title_text="Time", row=4, col=1)
    
    # ------------------------------------------------------------------
    # Save & Display
    # ------------------------------------------------------------------
    if save_html:
        filename = fig_name + ".html"
        fig.write_html(FIGURES_DIR / filename,
                       include_plotlyjs="cdn")
    
    if show_figure:
        fig.show()

    return fig
##
##--------------------------------------

In [25]:
fig = plot_caes_timeseries(df_clean, "Clean synthetic dataset", 
                           save_html=SAVE_HTML, 
                           fig_name="clean_df_signals",
                           show_figure=SHOW_FIGURE)

In [26]:
fig = plot_caes_timeseries(df_noisy, "Noisy synthetic dataset", 
                           save_html=SAVE_HTML, 
                           fig_name="noisy_df_signals",
                           show_figure=SHOW_FIGURE)

In [27]:
dataset_file = RAW_DATA_DIR / "caes_dataset_clean.csv"

df_clean.to_csv(
    dataset_file,
    index=True
)

print(f"Dataset saved to:\n{dataset_file}")

Dataset saved to:
c:\Users\ma_gatti\Documents\Python\EnergyAnalyticsFramework\data\raw\caes_dataset_clean.csv


In [28]:
dataset_file = RAW_DATA_DIR / "caes_dataset_noisy.csv"

df_noisy.to_csv(
    dataset_file,
    index=True
)

print(f"Dataset saved to:\n{dataset_file}")

Dataset saved to:
c:\Users\ma_gatti\Documents\Python\EnergyAnalyticsFramework\data\raw\caes_dataset_noisy.csv
